# Day 38: Planning a Capstone

**Today's question:** how do you get from a vague idea ("an app for my recipes") to a plan you can build in three
days, and a program that runs from the very first hour?

We plan a **sample** project together: a **Recipe Box**. Your own capstone follows the same steps with
`PLAN-TEMPLATE.md`.

## 1. From idea to problem statement

**Idea:** "an app for my recipes". Too vague to build: *whose* recipes, and what is the *problem*?

**Problem statement:** *Diego cooks every evening and keeps recipes on scraps of paper. He forgets what he can cook
with the food already in his fridge, so food gets wasted. The Recipe Box stores his recipes and finds the ones that
use an ingredient he has.*

**User:** one person, on their own laptop, comfortable typing short answers. Not a chef, not a restaurant.

A good problem statement names **a person**, **a problem** and **how the app helps**. Everything we plan must serve it.

## 2. User stories, then the MVP

A **user story** describes one thing the user can do: *As a … I want … so that …*. The "so that" part stops us
building features nobody needs.

In [ ]:
stories = [
    {"story": "As a cook, I want to add a recipe with its ingredients so that I don't lose it", "priority": "Must"},
    {"story": "As a cook, I want to see all my recipes so that I can choose one", "priority": "Must"},
    {"story": "As a cook, I want to find recipes that use an ingredient so that food isn't wasted", "priority": "Must"},
    {"story": "As a cook, I want my recipes saved so that they are still there tomorrow", "priority": "Must"},
    {"story": "As a cook, I want to see quick recipes (under 30 min) for busy evenings", "priority": "Should"},
    {"story": "As a cook, I want to scale a recipe to more servings for guests", "priority": "Could"},
    {"story": "As a cook, I want to rate recipes so that I remember the good ones", "priority": "Could"},
    {"story": "As a cook, I want a photo of every dish", "priority": "Won't"},
]
print(len(stories), "stories")

The **MVP** (minimum viable product) is the smallest version that is already useful: only the **Must** stories.
🤔 **Predict:** how many stories are in the MVP?

In [ ]:
mvp = [item["story"] for item in stories if item["priority"] == "Must"]
for number, story in enumerate(mvp, start=1):
    print(number, story)

- "Saved so that they are still there tomorrow" is a **Must**: an app that forgets everything is not useful.
- "Photos" is **Won't**: images are out of scope for this course. Writing it down is still useful, because it ends
  the discussion.
- Should and Could stories wait until the MVP works **and is tested**.

## 3. Choosing the data format

Before writing code, write **one record** exactly as the program will store it.

In [ ]:
import csv
import json
from pathlib import Path

out_dir = Path("output")
out_dir.mkdir(exist_ok=True)

pancakes = {"name": "Pancakes", "servings": 4, "minutes": 20, "ingredients": ["flour", "eggs", "milk"]}
print(json.dumps(pancakes, indent=2))

A recipe contains a **list** of ingredients, and every recipe has a different number of them. Let's see what CSV
does with that.

In [ ]:
with open(out_dir / "recipes.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "servings", "minutes", "ingredients"])
    writer.writeheader()
    writer.writerow(pancakes)

print((out_dir / "recipes.csv").read_text(encoding="utf-8"))

🤔 **Predict:** we read the row back. What type is `ingredients` now?

In [ ]:
with open(out_dir / "recipes.csv", newline="", encoding="utf-8") as f:
    row = list(csv.DictReader(f))[0]
print(type(row["ingredients"]), row["ingredients"][0])

The list came back as **one string** (its first character is `[`). JSON keeps the list a list:

In [ ]:
with open(out_dir / "recipes.json", "w", encoding="utf-8") as f:
    json.dump([pancakes], f, indent=2)
with open(out_dir / "recipes.json", encoding="utf-8") as f:
    loaded = json.load(f)
print(type(loaded[0]["ingredients"]), loaded[0]["ingredients"][0])

**Decision for the plan:** `recipes.json`, a list of recipe dicts. Missing file (first run) → start with an empty
list. Damaged file → warning, start empty.

## 4. Nouns become classes, verbs become methods and functions

Read the Must stories again and underline **nouns** (things) and **verbs** (actions):

*add a **recipe** with its **ingredients** · see all **recipes** · find recipes that use an **ingredient** · recipes
**saved***

- **Recipe** has data *and* rules (a name cannot be empty, servings must be at least 1) → a **class**.
- An ingredient is just a word → a `str` in a list, no class needed.
- **add / find / save** → methods or functions.

In [ ]:
class Recipe:
    """One recipe: a name, servings, preparation time and a list of ingredients."""

    def __init__(self, name, servings, minutes):
        if name.strip() == "":
            raise ValueError("A recipe needs a name.")
        if servings < 1:
            raise ValueError("Servings must be at least 1.")
        self.name = name.strip()
        self.servings = servings
        self.minutes = minutes
        self.ingredients = []

    def add_ingredient(self, ingredient):
        """Add one ingredient (stored in lowercase). Refuses empty text and duplicates."""
        ingredient = ingredient.strip().lower()
        if ingredient == "" or ingredient in self.ingredients:
            raise ValueError(f"Cannot add ingredient '{ingredient}'.")
        self.ingredients.append(ingredient)

    def uses(self, ingredient):
        """Return True if this recipe needs the ingredient."""
        return ingredient.strip().lower() in self.ingredients

    def to_dict(self):
        """Return the recipe as a plain dict that json can save."""
        return {"name": self.name, "servings": self.servings, "minutes": self.minutes,
                "ingredients": self.ingredients}

    def __str__(self):
        return f"{self.name} ({self.servings} servings, {self.minutes} min): {', '.join(self.ingredients)}"

In [ ]:
dinner = Recipe("Pancakes", 4, 20)
for item in ["Flour", "eggs ", "milk"]:
    dinner.add_ingredient(item)
print(dinner)
print(dinner.uses("EGGS"), dinner.uses("rice"))
print(dinner.to_dict())

**Modules** (each with one job). This table goes straight into the plan:

| File | Job | Contains |
|------|-----|----------|
| `models.py` | the data and its rules | `Recipe` |
| `storage.py` | files | `load_recipes(path)`, `save_recipes(recipes, path)`, `recipe_from_dict(data)` |
| `search.py` | answering questions (no input, no print) | `find_by_ingredient(recipes, ingredient)`, `quick_recipes(recipes, max_minutes)` |
| `main.py` | talking to the user | `main()`, `ask_text()`, `ask_int()`, one function per menu option |

## 5. A walking skeleton

A **walking skeleton** is the whole program with nothing finished: every module and menu option exists, but each
one only says `TODO`. It runs from the first hour, so every later step is "replace one `TODO`, run, test".

New keyword: **`pass`** means "do nothing". Python needs *something* inside a function body; `pass` is a placeholder.

In [ ]:
def load_recipes(path):
    """Return the list of Recipe objects saved at path ([] if the file does not exist)."""
    return []   # TODO: read JSON (step 4 of the build plan)


def save_recipes(recipes, path):
    """Save all recipes to path as JSON."""
    pass        # TODO


def handle_choice(choice, recipes):
    """Run one menu option and return the message to show the user."""
    if choice == "1":
        return f"TODO: list {len(recipes)} recipes"
    elif choice == "2":
        return "TODO: add a recipe"
    elif choice == "3":
        return "TODO: find by ingredient"
    elif choice == "0":
        return "Goodbye!"
    return f"'{choice}' is not on the menu."

`handle_choice` **returns** a message instead of calling `input()` itself, so we can try the menu without typing.
In `main.py`, the loop gets `choice` from `input()` and prints the result.
🤔 **Predict** the four messages.

In [ ]:
recipes = load_recipes(out_dir / "recipes.json")
for choice in ["1", "3", "9", "0"]:
    print(choice, "->", handle_choice(choice, recipes))

## 6. Write the test before the function

For each function in the plan, decide what a correct answer looks like **before** writing it. A stub that is not
built yet fails its test: that is fine and expected ("red").

In [ ]:
def find_by_ingredient(recipes, ingredient):
    """Return the recipes that use this ingredient (upper/lower case does not matter)."""
    return []   # TODO


def test_find_by_ingredient():
    salad = Recipe("Salad", 2, 10)
    salad.add_ingredient("tomato")
    assert find_by_ingredient([dinner, salad], "EGGS") == [dinner]
    assert find_by_ingredient([dinner, salad], "rice") == []

In [ ]:
test_find_by_ingredient()

Now build the function until the test passes ("green").

In [ ]:
def find_by_ingredient(recipes, ingredient):
    """Return the recipes that use this ingredient (upper/lower case does not matter)."""
    return [recipe for recipe in recipes if recipe.uses(ingredient)]


test_find_by_ingredient()
print("test passed")

In the real project this test lives in `test_search.py` and runs with `uv run pytest -q`.

## Recap

1. **Problem statement**: a person, a problem, how the app helps.
2. **User stories** (*As a … I want … so that …*), each marked Must / Should / Could / Won't. **MVP** = the Musts,
   and saving is always a Must.
3. Choose the **data format** by writing one real record: nested → JSON, flat table → CSV.
4. **Nouns → classes, verbs → methods/functions**; one job per module.
5. Build a **walking skeleton** first, and write tests before (or right after) each function.

**Next:** fill in your own `PLAN.md`, get it approved, and leave today with a skeleton that runs.